# Aula 2 — Setup e Dados

**Intensivão Quant AI — ImpactUFSCar**

---

Hoje você vai ter pela primeira vez os dados que vão alimentar toda a estratégia. Ao final desta aula, você terá:

- Um DataFrame com retornos diários de todas as ações do IBOVESPA
- Entendido a diferença entre retorno simples e log
- Os primeiros plots — curva de preço e curva de retorno

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. O que é uma série temporal de preços?

Uma **série temporal** é simplesmente uma sequência de dados coletados e ordenados ao longo do tempo. No mercado financeiro, a série temporal de preços é a nossa matéria-prima fundamental. Cada linha representa uma observação (por exemplo, o preço de fechamento de uma ação) em uma data específica.

Imagine uma planilha simples:
```
Data         | PETR4 (Preço em R$)
2024-01-02   | 37.50
2024-01-03   | 38.10
2024-01-04   | 37.80
```

### O Grande Problema dos Preços Brutos (Nominais)

À primeira vista, olhar para os preços de R$ 37.50 ou R$ 38.10 parece natural. No entanto, para fins de análise quantitativa e estatística, **preços brutos são péssimos**. Por quê?
1. **Falta de comparabilidade:** WEGE3 custando R$ 45.00 e PETR4 custando R$ 37.00 não nos diz absolutamente nada sobre qual ativo performou melhor. Se WEGE3 subir R$ 1.00 e PETR4 subir R$ 1.00, a variação percentual é completamente diferente.
2. **Não-estacionariedade:** Preços brutos tendem a crescer ou cair estruturalmente ao longo do tempo (têm tendência). Isso faz com que a média e a variância dos preços mudem de acordo com a janela escolhida, violando princípios estatísticos básicos.
3. **Escala diferente:** Empresas diferentes têm preços nominais muito diferentes devido à quantidade de ações em circulação.

**Solução Quantitativa:** Nós nunca modelamos preços brutos diretamente! Em vez disso, transformamos os preços em **retornos**. O retorno representa a taxa de crescimento percentual do capital investido naquele ativo.


## 2. Retorno Simples vs. Retorno Logarítmico

Para calcular o retorno de um ativo entre o período $t-1$ e o período $t$, temos duas abordagens principais na literatura financeira. Vamos entender a matemática de cada uma de forma bem mastigada.

---

### A. Retorno Simples ($R_t$)

O **retorno simples** é a variação percentual que você já conhece no dia a dia. É a diferença de preço dividida pelo preço inicial:

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

Onde:
- $P_t$ é o preço no instante atual (ex: fechamento de hoje).
- $P_{t-1}$ é o preço no instante anterior (ex: fechamento de ontem).

**Intuição:** Se uma ação custava R$ 100 ontem e passou a custar R$ 105 hoje, o retorno simples é:
$$R_t = \frac{105 - 100}{100} = 0.05 = 5\%$$

---

### B. Retorno Logarítmico ou Contínuo ($r_t$)

O **retorno logarítmico** (também chamado de *log-return*) utiliza o logaritmo natural ($\ln$) da razão de preços:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

**Intuição:** No mesmo exemplo (de R$ 100 para R$ 105):
$$r_t = \ln\left(\frac{105}{100}\right) = \ln(1.05) \approx 0.04879 = 4.88\%$$

---

### Por que os Quants preferem o Retorno Logarítmico? (A Propriedade Mágica)

À primeira vista, o retorno logarítmico parece mais complicado. Porém, ele possui propriedades matemáticas fantásticas para análise quantitativa:

#### 1. Aditividade Temporal (A Soma Telescópica)
Se você quer saber o retorno acumulado de um ativo durante 3 dias (dias 1, 2 e 3):
- Com **retorno simples**, você precisa **multiplicar** os fatores de retorno:
  $$1 + R_{\text{acumulado}} = (1 + R_1)(1 + R_2)(1 + R_3) - 1$$
- Com **retorno logarítmico**, basta **somar** os retornos diários!
  $$r_{\text{acumulado}} = r_1 + r_2 + r_3$$

**Prova Matemática:**
$$r_1 + r_2 + r_3 = \ln\left(\frac{P_1}{P_0}\right) + \ln\left(\frac{P_2}{P_1}\right) + \ln\left(\frac{P_3}{P_2}\right)$$
Aplicando a propriedade dos logaritmos $\ln(a) + \ln(b) = \ln(a \cdot b)$:
$$r_1 + r_2 + r_3 = \ln\left(\frac{P_1}{P_0} \cdot \frac{P_2}{P_1} \cdot \frac{P_3}{P_2}\right) = \ln\left(\frac{P_3}{P_0}\right) = r_{\text{acumulado}}$$
Os termos intermediários se cancelam (uma soma telescópica)! Isso torna a manipulação algébrica e computacional infinitamente mais simples e rápida.

#### 2. Relação e Aproximação
Para retornos diários muito pequenos, os dois retornos são praticamente idênticos. Isso ocorre porque a expansão de série de Taylor de $\ln(1 + x)$ para um $x$ muito pequeno é:
$$\ln(1 + x) \approx x - \frac{x^2}{2} + \dots \approx x$$
Como os retornos diários costumam ser da ordem de $0.1\%$ a $2\%$, a diferença prática entre os dois é insignificante no curto prazo, mas no longo prazo as curvas divergem significativamente.


In [ ]:
# Baixamos PETR4 como exemplo para demonstrar os dois tipos de retorno
petr4 = yf.download("PETR4.SA", start="2020-01-01", end="2024-12-31",
                    auto_adjust=True, progress=False)

preco = petr4['Close'].squeeze()

# Calculando os dois retornos
ret_simples = preco.pct_change()
ret_log     = np.log(preco / preco.shift(1))

print("Primeiros 5 dias:")
comparacao = pd.DataFrame({'ret_simples': ret_simples, 'ret_log': ret_log})
print(comparacao.dropna().head())
print(f"\nDiferença média (absoluta): {(ret_simples - ret_log).abs().mean():.6f}")

In [ ]:
# Visualizando: no curto prazo são praticamente iguais
# No longo prazo, o retorno acumulado diverge

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Retornos diários — quase indistinguíveis
ret_simples.dropna().iloc[:60].plot(ax=axes[0], label='simples', alpha=0.8)
ret_log.dropna().iloc[:60].plot(ax=axes[0], label='log', alpha=0.8, linestyle='--')
axes[0].set_title('Retornos diários — 60 primeiros dias')
axes[0].legend()

# Retorno acumulado — divergência aparece
acum_simples = (1 + ret_simples).cumprod() - 1
acum_log     = ret_log.cumsum()

acum_simples.plot(ax=axes[1], label='simples acumulado')
acum_log.plot(ax=axes[1], label='log acumulado', linestyle='--')
axes[1].set_title('Retorno acumulado — 2020 a 2024')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Retorno simples acumulado:", f"{acum_simples.iloc[-1]:.2%}")
print("Retorno log acumulado:    ", f"{np.exp(acum_log.iloc[-1]) - 1:.2%}  (convertido de volta para %)")

## 3. Preços Ajustados — Proventos (Dividendos) e Desdobramentos (Splits)

Quando baixamos dados históricos de preços de ações, precisamos tomar um cuidado extremo para não colocar "ruído" ou mentiras nos nossos modelos. Há dois eventos corporativos principais que distorcem os preços históricos brutos:

---

### A. Desdobramento (Split) e Grupamento (Inplit)
- **O que é:** Uma empresa decide multiplicar seu número de ações para torná-las mais baratas e líquidas no mercado. Por exemplo, num desdobramento de **1 para 2**, quem tinha 1 ação cotada a R$ 100 passa a ter 2 ações cotadas a R$ 50. O valor total do investidor continua sendo R$ 100.
- **O perigo no gráfico bruto:** Se você olhar o preço bruto histórico, verá uma queda repentina de R$ 100 para R$ 50 (uma queda de $50\%$) de um dia para o outro. Mas **essa queda não representa perda real de patrimônio!** Se seu algoritmo de trading usar preços brutos, ele registrará uma perda falsa monstruosa e tomará decisões erradas.

---

### B. Distribuição de Dividendos e JCP (Juros sobre Capital Próprio)
- **O que é:** As empresas distribuem parte de seus lucros em dinheiro para os acionistas. Na chamada **data ex-dividendo**, o preço da ação no mercado sofre um desconto automático equivalente ao valor do dividendo que foi pago.
- **O perigo no gráfico bruto:** Assim como no split, o preço bruto cai repentinamente. No entanto, o investidor recebeu esse dinheiro diretamente em sua conta bancária. Tratar essa queda como um retorno negativo é um erro metodológico grave.

---

### A Solução Quant: O Preço Ajustado
Para eliminar esses saltos artificiais e garantir que os retornos representem a **rentabilidade real total do investidor (Total Return)**, nós ajustamos a série histórica de preços retroativamente.
- Os preços históricos são multiplicados por um fator para remover o impacto de desdobramentos e dividendos passados.
- No Python, a biblioteca `yfinance` faz esse ajuste automaticamente de forma brilhante se passarmos o parâmetro `auto_adjust=True`.

> **Regra de Ouro:** NUNCA calcule retornos ou faça backtests usando preços históricos brutos (nominais). Use sempre **preços de fechamento ajustados**.


In [ ]:
# Demonstração: preço bruto vs ajustado para VALE3
vale_bruto    = yf.download("VALE3.SA", start="2020-01-01", end="2024-12-31",
                             auto_adjust=False, progress=False)['Close'].squeeze()
vale_ajustado = yf.download("VALE3.SA", start="2020-01-01", end="2024-12-31",
                             auto_adjust=True,  progress=False)['Close'].squeeze()

fig, ax = plt.subplots(figsize=(12, 4))
vale_bruto.plot(ax=ax, label='preço bruto', alpha=0.7)
vale_ajustado.plot(ax=ax, label='preço ajustado', alpha=0.7)
ax.set_title('VALE3 — Bruto vs Ajustado (dividendos incluídos no ajustado)')
ax.legend()
plt.show()

## 4. O Universo IBOVESPA e o Viés de Sobrevivência (Survivorship Bias)

O **IBOVESPA** é o principal índice da bolsa brasileira (B3). Ele é um índice de retorno total ponderado pelo valor de mercado de free float (ações em circulação) das empresas mais negociadas.
A carteira teórica do IBOVESPA é rebalanceada a cada quatro meses (quadrimestralmente) para garantir que apenas as ações mais líquidas façam parte do índice.

---

### Um Conceito Avançado para Iniciantes: Survivorship Bias (Viés de Sobrevivência)

Quando estamos fazendo pesquisas quantitativas, é muito comum cometermos um erro clássico sem perceber: **usar a composição atual do IBOVESPA para fazer um backtest de 10 anos atrás**.

#### Por que isso é perigoso?
Se pegarmos a lista de empresas que compõem o IBOVESPA *hoje* e baixarmos o histórico delas desde 2012, estaremos selecionando apenas empresas que **sobreviveram e prosperaram** até hoje.
- Nós ignoramos todas as empresas que faziam parte do IBOVESPA em 2012, mas que ao longo do caminho faliram, foram deslistadas ou perderam muita liquidez (por exemplo, OGX, Americanas, PDG, etc.).
- Como resultado, nosso backtest terá um resultado artificialmente maravilhoso (enviesado para cima), pois removemos os "perdedores" do passado de forma retroativa.
- Na indústria de investimentos real, para evitar o Viés de Sobrevivência, utilizamos **bases de dados de carteira histórica (point-in-time)**, onde em cada mês do passado compramos exatamente as ações que compunham o índice *naquele momento do passado*.

Neste intensivão, para manter o código limpo e acessível para iniciantes, usaremos uma carteira simplificada focada nas principais componentes líquidas históricas, mas é de extrema importância que você documente e mencione o **Viés de Sobrevivência** no seu relatório para a banca do Itaú! Isso demonstrará um rigor técnico digno de quants experientes.


In [ ]:
# Componentes do IBOVESPA (principais ativos, sufixo .SA para B3)
TICKERS_IBOV = [
    'ABEV3.SA', 'ASAI3.SA', 'AZUL4.SA', 'B3SA3.SA', 'BBAS3.SA',
    'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BPAC11.SA', 'BRAP4.SA',
    'BRFS3.SA', 'BRKM5.SA', 'CASH3.SA', 'CCRO3.SA', 'CIEL3.SA',
    'CMIG4.SA', 'CMIN3.SA', 'COGN3.SA', 'CPFE3.SA', 'CPLE6.SA',
    'CSAN3.SA', 'CSNA3.SA', 'CYRE3.SA', 'DXCO3.SA', 'EGIE3.SA',
    'ELET3.SA', 'ELET6.SA', 'EMBR3.SA', 'ENEV3.SA', 'ENGI11.SA',
    'EQTL3.SA', 'EZTC3.SA', 'FLRY3.SA', 'GGBR4.SA', 'GOAU4.SA',
    'GOLL4.SA', 'HAPV3.SA', 'HYPE3.SA', 'IGTI11.SA', 'IRBR3.SA',
    'ITSA4.SA', 'ITUB4.SA', 'JBSS3.SA', 'KLBN11.SA', 'LREN3.SA',
    'LWSA3.SA', 'MGLU3.SA', 'MRFG3.SA', 'MRVE3.SA', 'MULT3.SA',
    'NTCO3.SA', 'PCAR3.SA', 'PETR3.SA', 'PETR4.SA', 'PRIO3.SA',
    'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RDOR3.SA', 'RENT3.SA',
    'RRRP3.SA', 'SANB11.SA', 'SBSP3.SA', 'SLCE3.SA', 'SMTO3.SA',
    'SULA11.SA', 'SUZB3.SA', 'TAEE11.SA', 'TIMS3.SA', 'TOTS3.SA',
    'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VBBR3.SA', 'VIVT3.SA',
    'WEGE3.SA', 'YDUQ3.SA',
]

print(f"Total de tickers: {len(TICKERS_IBOV)}")

In [ ]:
# Download dos preços ajustados — pode levar 1-2 minutos
DATA_INICIO = '2010-01-01'
DATA_FIM    = '2024-12-31'

print("Baixando preços... (aguarde)")
precos_raw = yf.download(
    TICKERS_IBOV,
    start=DATA_INICIO,
    end=DATA_FIM,
    auto_adjust=True,
    progress=True
)['Close']

print(f"\nShape: {precos_raw.shape}  →  {precos_raw.shape[0]} dias × {precos_raw.shape[1]} ações")
print(f"Período: {precos_raw.index[0].date()} a {precos_raw.index[-1].date()}")

## 5. Entendendo o DataFrame

O `precos_raw` é um DataFrame com:
- **Linhas:** dias úteis
- **Colunas:** ações
- **Valores:** preço de fechamento ajustado em reais

Antes de calcular qualquer coisa, precisamos inspecionar os dados.

In [ ]:
# Inspecionando o DataFrame
print("Primeiras linhas:")
print(precos_raw.iloc[:3, :6])  # 3 linhas, 6 primeiras colunas

print("\nÚltimas linhas:")
print(precos_raw.iloc[-3:, :6])

In [ ]:
# Dados ausentes — quantos NaNs por coluna?
nans = precos_raw.isna().sum().sort_values(ascending=False)
print("Ações com mais dados ausentes:")
print(nans[nans > 0].head(10))
print(f"\nTotal de ações sem nenhum NaN: {(nans == 0).sum()}")

**Por que existem NaNs?** Algumas ações do IBOVESPA atual foram listadas depois de 2010 (IPOs). Outras tiveram períodos de suspensão. Na Aula 3 vamos tratar isso com mais cuidado — por hora, seguimos em frente.

## 6. Calculando os retornos

O DataFrame de preços vira um DataFrame de **retornos logarítmicos diários**. Essa será nossa matéria-prima para todo o restante do intensivão.

In [ ]:
# Retornos log diários
# np.log(P_t / P_{t-1}) = np.log(P_t) - np.log(P_{t-1})
retornos = np.log(precos_raw / precos_raw.shift(1))

print(f"Shape retornos: {retornos.shape}")
print(f"NaNs na primeira linha (esperado): {retornos.iloc[0].isna().all()}")

# Removemos a primeira linha (toda NaN — não há retorno no dia 0)
retornos = retornos.iloc[1:]

print(f"\nEstatísticas básicas de PETR4:")
print(retornos['PETR4.SA'].describe())

## 7. Janelas de tempo

No intensivão vamos trabalhar com três frequências:

| Frequência | Uso | Como calcular |
|---|---|---|
| Diária | EDA, visualização | `retornos` (já temos) |
| Mensal | Sinal de momentum, backtest | `retornos.resample('ME').sum()` |
| Anual | Métricas de desempenho | `retornos.resample('YE').sum()` |

Para o **sinal de momentum** (aula 4) vamos usar retornos mensais. Para o **backtest** (aula 5) rebalanceamos mensalmente. Por isso, vale já preparar os dois DataFrames.

In [ ]:
# Retornos mensais — soma dos retornos log diários dentro de cada mês
# (lembre: log é aditivo, então somar é correto)
retornos_mensais = retornos.resample('ME').sum()

print(f"Retornos mensais shape: {retornos_mensais.shape}")
print(f"Período: {retornos_mensais.index[0].date()} a {retornos_mensais.index[-1].date()}")
print()
print("Últimos 3 meses de PETR4 e VALE3:")
print(retornos_mensais[['PETR4.SA', 'VALE3.SA']].tail(3))

## 8. Primeiros plots

Antes de modelar qualquer coisa, olhamos para os dados. Dois plots fundamentais: o preço de uma ação no tempo, e os retornos dela.

In [ ]:
# Plot 1: preço ajustado de algumas ações
acoes_exemplo = ['PETR4.SA', 'VALE3.SA', 'WEGE3.SA', 'ITUB4.SA']

fig, ax = plt.subplots(figsize=(13, 5))

for ticker in acoes_exemplo:
    # Normalizamos para 100 em 2010 para comparar na mesma escala
    serie = precos_raw[ticker].dropna()
    (serie / serie.iloc[0] * 100).plot(ax=ax, label=ticker.replace('.SA', ''))

ax.set_title('Preço ajustado — normalizado em 100 (Jan/2010)')
ax.set_ylabel('Índice (base 100)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: retornos diários de PETR4 — parece ruído, mas tem estrutura
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

retornos['PETR4.SA'].plot(ax=axes[0], alpha=0.7, color='steelblue')
axes[0].set_title('Retornos diários — PETR4 (2010–2024)')
axes[0].set_ylabel('Retorno log')
axes[0].axhline(0, color='black', linewidth=0.8)

retornos_mensais['PETR4.SA'].plot(ax=axes[1], kind='bar',
                                   alpha=0.7, color='steelblue',
                                   width=0.8)
axes[1].set_title('Retornos mensais — PETR4 (2010–2024)')
axes[1].set_ylabel('Retorno log mensal')
axes[1].set_xticklabels(
    [d.strftime('%Y-%m') if i % 12 == 0 else ''
     for i, d in enumerate(retornos_mensais.index)],
    rotation=45
)

plt.tight_layout()
plt.show()

## 9. Salvando os dados

Vamos salvar os DataFrames para não precisar baixar tudo de novo nas próximas aulas.

In [ ]:
precos_raw.to_parquet(os.path.join(DADOS_DIR, 'precos_ibov.parquet'))
retornos.to_parquet(os.path.join(DADOS_DIR, 'retornos_diarios.parquet'))
retornos_mensais.to_parquet(os.path.join(DADOS_DIR, 'retornos_mensais.parquet'))

print("Arquivos salvos:")
print("  precos_ibov.parquet       — preços ajustados diários")
print("  retornos_diarios.parquet  — retornos log diários")
print("  retornos_mensais.parquet  — retornos log mensais")

## Resumo da aula

| O que fizemos | Por que importa |
|---|---|
| Retorno log vs simples | Log é aditivo — facilita cálculos acumulados e é o padrão em finanças quant |
| `auto_adjust=True` | Sem ajuste, dividendos e splits criam retornos falsos |
| DataFrame de retornos | É a matéria-prima de toda a estratégia |
| NaNs identificados | Ações com histórico mais curto — tratar na aula seguinte |
| Retornos mensais | Frequência de rebalanceamento da estratégia de momentum |

---

## Para replicar em casa

1. Rode este notebook do início ao fim
2. Verifique que os três arquivos `.parquet` foram criados
3. Explore: quais ações tiveram o maior e menor retorno acumulado no período?

```python
# Dica: retorno acumulado de cada ação
retorno_acumulado = retornos.sum().sort_values(ascending=False)
print(retorno_acumulado.head(5))   # maiores
print(retorno_acumulado.tail(5))   # menores
```

---

*ImpactUFSCar — Diretoria de Quant*